## Email Ingestion LLM

This notebook applies a suitable LLM model from an active Groq key, to help ingest semi-structured underwriting emails. 

In [1]:
import getpass
import json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

# Load your Groq key
load_dotenv()
# Prompt for the key interactively via a masked input box if not already set
api_key = getpass.getpass("Enter your Groq API key: ")
client = Groq(api_key=api_key)
MODEL_NAME = "openai/gpt-oss-20b"

In [2]:

def default_data_directory():
    """Find the repository sample-email directory."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        data_directory = folder / "data" / "email_ingestion" / "sample emails"
        if data_directory.is_dir():
            return data_directory
    raise FileNotFoundError(f"Sample emails not found from {Path.cwd()}")


def group_submission_files(directory=None, submission_ids=None):
    """Combine each email and its attachments into one payload."""
    directory = Path(directory) if directory else default_data_directory()
    files = sorted(directory.glob("*.md"))
    available_ids = {file.name.split("_")[0] for file in files}
    selected_ids = set(submission_ids) if submission_ids is not None else available_ids

    missing_ids = selected_ids - available_ids
    if missing_ids:
        raise ValueError(f"Submission IDs not found: {sorted(missing_ids)}")

    submissions = {}
    for file in files:
        submission_id = file.name.split("_")[0]
        if submission_id in selected_ids:
            submissions.setdefault(submission_id, "")
            submissions[submission_id] += (
                f"\n\n--- FILE: {file.name} ---\n"
                f"{file.read_text(encoding='utf-8')}"
            )
    return submissions


def apply_confidence_rules(data, source_text):
    """Set confidence and review fields from clear source conflicts."""
    text = source_text.casefold()
    low_reasons = []
    medium_reasons = []

    if (("standalone" in text or "legal entity" in text)
            and ("group" in text or "consolidated" in text)):
        low_reasons.append("Group and standalone information are both present.")

    if (("estimate" in text or "estimated" in text)
            and ("audited" in text or "actual" in text)):
        low_reasons.append("Estimated and audited/actual figures are both present.")

    if "expiring" in text and "requested" in text:
        medium_reasons.append("Expiring and newly requested cover are both mentioned.")

    if (("primary address" in text or "registered office" in text)
            and ("operating" in text or "territor" in text)):
        medium_reasons.append("Address and operating-territory countries may differ.")

    reasons = data.get("review_reasons", []) + low_reasons + medium_reasons
    data["review_reasons"] = list(dict.fromkeys(reasons))
    data["confidence"] = "Low" if low_reasons else "Medium" if medium_reasons else "High"
    data["review_required"] = data["confidence"] != "High"
    return data


def extract_submission_data(submission_id, source_text):
    """Ask the LLM to extract fields and explain any source ambiguity."""
    prompt = (
        "Extract the underwriting fields from the email and attachments. Return only valid JSON.\n\n"
        "Compare every document before choosing a value. Confidence is based on source agreement, not ground truth.\n"
        "Use High only when every core field is explicit and consistent. Use Medium for a plausible alternative interpretation. "
        "Use Low when documents conflict or the correct basis cannot be determined. Set review_required true for Medium or Low.\n\n"
        "Use last completed actual revenue, not projected revenue. Distinguish standalone/group, audited/estimated, "
        "operating countries/domicile, and requested/expiring cover. Do not invent missing values.\n\n"
        "Return exactly this JSON schema:\n"
        "{\n"
        '  "submission_id": "string", "company_name": "string or null",\n'
        '  "revenue": integer or null, "countries": [],\n'
        '  "industry": "string or null", "requested_coverages": [],\n'
        '  "confidence": "High|Medium|Low", "review_required": true,\n'
        '  "review_reasons": [], "explanation": "brief evidence-based explanation"\n'
        "}"
    )

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0.0,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Extract this submission:\n{source_text}"},
            ],
        )
        raw_output = response.choices[0].message.content.strip()
        raw_output = raw_output.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw_output)
        data["submission_id"] = submission_id
        return apply_confidence_rules(data, source_text)
    except Exception as error:
        return {
            "submission_id": submission_id,
            "confidence": "Low",
            "review_required": True,
            "review_reasons": [f"Extraction failed: {error}"],
            "explanation": "The submission needs manual review.",
        }

In [3]:
# Select IDs to test, or set to None to process all submissions.
# Example: TEST_SUBMISSION_IDS = ["E007", "E014"]
TEST_SUBMISSION_IDS = None

submission_docs = group_submission_files(submission_ids=TEST_SUBMISSION_IDS)

In [56]:
# Run extraction pipeline
import time

extracted_records = []
for sub_id, payload in submission_docs.items():
    print(f"Processing {sub_id}...")
    record = extract_submission_data(sub_id, payload)
    extracted_records.append(record)
    
    # 3-second delay guarantees < 8000 tokens per minute
    if len(extracted_records) > 20:
        time.sleep(3)

Processing E001...
Processing E002...
Processing E003...
Processing E004...
Processing E005...
Processing E006...
Processing E007...
Processing E008...
Processing E009...
Processing E010...
Processing E011...
Processing E012...
Processing E013...
Processing E014...
Processing E015...
Processing E016...
Processing E017...
Processing E018...
Processing E019...
Processing E020...
Processing E021...
Processing E022...
Processing E023...
Processing E024...
Processing E025...
Processing E026...
Processing E027...
Processing E028...
Processing E029...
Processing E030...
Processing E031...
Processing E032...
Processing E033...
Processing E034...
Processing E035...
Processing E036...
Processing E037...
Processing E038...
Processing E039...
Processing E040...
Processing E041...
Processing E042...
Processing E043...
Processing E044...
Processing E045...
Processing E046...
Processing E047...
Processing E048...
Processing E049...
Processing E050...


In [58]:
df = pd.DataFrame(extracted_records)

# Keep only the agreed evaluation and review columns.
columns = [
    "submission_id", "company_name", "revenue", "countries", "industry",
    "requested_coverages", "confidence", "review_required", "review_reasons",
    "explanation",
]
df = df.reindex(columns=columns)

output_path = default_data_directory().parent / "llm_extracted_submissions.csv"
df.to_csv(output_path, index=False)

display(df)

,submission_id,company_name,revenue,countries,industry,requested_coverages,confidence,review_required,review_reasons,explanation
0,E001,Everstead Pharma Ltd,14608818,"[Sweden, Canada, Singapore]",Manufacturing,"[Property, Kidnap & Ransom]",High,False,[],"Revenue, operating countries, industry, and re..."
1,E002,Crestline Textiles Ltd,38122777,"[Spain, Norway, Belgium]",Technology,"[General Liability, Property]",Medium,True,[Expiring and newly requested cover are both m...,All requested fields are explicitly stated in ...
2,E003,Granite Analytics Ltd,7808428,[United States],Healthcare,"[Professional Indemnity, Technology E&O, Direc...",Medium,True,[Expiring and newly requested cover are both m...,"Revenue, industry, country, and requested cove..."
3,E004,Highland Logistics Ltd,35958262,"[United Arab Emirates, United Kingdom]",Fintech,"[Professional Indemnity, Product Liability, Ma...",High,False,[],"Revenue, countries, industry, and requested co..."
4,E005,Ironclad Trading Ltd,29626602,[Denmark],Retail,[Management Liability],Medium,True,[Address and operating-territory countries may...,"Revenue, country, industry, and requested cove..."
5,E006,Alder Engineering Ltd,42413185,"[Japan, Canada]",Logistics,[Media Liability],Medium,True,[Address and operating-territory countries may...,All core fields are explicitly stated and cons...
6,E007,Foxmere Technologies Ltd,488379,"[Sweden, Singapore, United Kingdom]",Construction,[Media Liability],High,False,[],"Revenue, countries, industry, and requested co..."
7,E008,Redwood Manufacturing Ltd,16614618,"[France, United Arab Emirates, Singapore]",Media,[Product Liability],Medium,True,[Address and operating-territory countries may...,"Revenue, countries, industry and requested cov..."
8,E009,Ivybridge Construction Ltd,37346694,[Spain],Life Sciences,"[General Liability, Technology E&O, Directors ...",Medium,True,[Address and operating-territory countries may...,All requested fields are explicitly stated in ...
9,E010,Eastwick Group Ltd,45526148,"[Switzerland, Denmark, Japan]",Professional Services,[General Liability],High,False,[],"Revenue, countries, industry, and requested co..."
